# Chat Template Testing and Tokenization Visualization Notebook

This notebook is used to test chat templates by applying them to conversation messages and visualizing the tokenization output with color-coded token highlighting.

**Use Cases:**
- Validate chat template formatting before training
- Debug assistant token masking for SFT
- Test tool calling template behavior
- Inspect tokenizer boundary handling and special tokens
- Verify thinking tags in reasoning templates

## Configuration

Edit these values to customize the tokenizer, chat template, and visualization options.

In [8]:
# Tokenizer settings
TOKENIZER_NAME_OR_PATH = "Polygl0t/Tucano2-0.6B-Base"  # Path to tokenizer model
CHAT_TEMPLATE_PATH = "./jinja_templates/chat_template_non_reasoning.jinja"  # Path to chat template file (set to None to use tokenizer's default)
MESSAGES_FILE = "./chat_sample.json"          # Path to JSON file containing messages (set to None for default messages)

# Template options
ADD_GENERATION_PROMPT = False              # Enable generation prompt
ENABLE_THINKING = False                    # Enable thinking mode

# Visualization options
SHOW_TEXT = True                           # Show decoded text output
SHOW_IDS = False                           # Show token IDs alongside tokens

## Apply Chat Template

Apply the chat template to tokenize the conversation and generate an assistant-token-masks.

In [5]:
import json
from transformers import AutoTokenizer
from rich.text import Text
from rich.console import Console
from rich.panel import Panel
import ftfy

console = Console()

def clean_token(token: str) -> str:
    """Normalize tokens from different tokenizer styles into readable text."""
    
    # Handle newline tokens
    token = token.replace("Ċ", "\n").replace("Ď", "\n")
    
    # Handle space prefixes
    if token.startswith("▁"):  # SentencePiece
        token = " " + token[1:]
    elif token.startswith("Ġ"):  # GPT/BPE
        token = " " + token[1:]
    
    # Handle escaped newline
    token = token.replace("<0x0A>", "\n")

    # Decode UTF-8 fallback artifacts / fix mojibake
    try:
        if any(bad in token for bad in ("Ã", "Â", "Ã¡", "Ã©", "Ãª", "Ã³", "Ã­", "Ã£", "Ãµ")):
            token = token.encode("latin1").decode("utf-8")
    except Exception:
        try:
            token = token.encode("utf-8").decode("latin1")
        except Exception:
            pass
    try:
        token = ftfy.fix_text(token)
    except Exception:
        pass
    
    # Remove stray byte prefixes
    token = token.replace("Â", "").replace("Ġ", " ").replace("▁", " ")
    
    return token

def load_tokenizer(tokenizer_name_or_path, template_path=None):
    """Load tokenizer and optionally set custom chat template."""
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path, use_fast=True)
    print(f"Loaded tokenizer from: {tokenizer_name_or_path}")
    
    if template_path:
        with open(template_path) as f:
            tokenizer.chat_template = f.read()
        print(f"Loaded chat template from: {template_path}")
    
    return tokenizer

def load_messages_and_tools(messages_file=None):
    """Load messages from JSON file or return default messages."""
    if messages_file:
        with open(messages_file) as f:
            data = json.load(f)[0]
            messages, tools = data.get("messages", []), data.get("tools", [])
            if messages or tools:
                print(f"Loaded messages and tools from: {messages_file}")
                return messages, tools
    
    # Default messages for testing
    data = {
        "messages": [
            {
                "role": "user",
                "content": "Hello!",
            },
            {
                "role": "assistant",
                "content": "Hello! How can I assist you today?",
            },
        ],
        "tools": [],
    }
    print("Using default messages and tools.")
    return data["messages"], data["tools"]

def visualize_tokens(tokenizer, tokenized_output, show_ids=False):
    """Visualize tokenized output with color coding."""
    text_visualization = Text()
    tokens = tokenizer.convert_ids_to_tokens(tokenized_output["input_ids"])
    cleaned_tokens = [clean_token(token) for token in tokens]

    for i, (token, mask) in enumerate(zip(cleaned_tokens, tokenized_output["assistant_masks"])):
        color = "green" if mask else "gray"
        if show_ids:
            token_id = tokenized_output["input_ids"][i]
            text_visualization.append(f"[{token_id}]{token}", style=color)
        else:
            text_visualization.append(token, style=color)
    
    return text_visualization

# Load tokenizer
tokenizer = load_tokenizer(TOKENIZER_NAME_OR_PATH, CHAT_TEMPLATE_PATH)
# Load messages and tools
messages, tools = load_messages_and_tools(MESSAGES_FILE)
# Apply chat template to tokenize messages
tokenized_output = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    return_assistant_tokens_mask=True,
    return_dict=True,
    add_generation_prompt=ADD_GENERATION_PROMPT,
    enable_thinking=ENABLE_THINKING,
)

Loaded tokenizer from: Polygl0t/Tucano2-0.6B-Base
Loaded chat template from: ./jinja_templates/chat_template_non_reasoning.jinja
Loaded messages and tools from: ./chat_sample.json


## Visualize Results

Display the decoded text and color-coded token visualization:
- **Green**: Assistant tokens (trained during SFT)
- **Gray**: Other tokens (masked during training)

In [9]:
# Show decoded text if requested
if SHOW_TEXT:
    decoded = tokenizer.decode(tokenized_output["input_ids"])
    console.print(Panel(decoded, title="Decoded Text", border_style="blue"))

# Visualize tokens
text_viz = visualize_tokens(tokenizer, tokenized_output, SHOW_IDS)
console.print(Panel(text_viz, title="Token Visualization (Green=Assistant, Gray=Other)", border_style="magenta"))
total_tokens = len(tokenized_output["input_ids"])
assistant_tokens = sum(tokenized_output["assistant_masks"])

print(f"\nTotal tokens: {total_tokens}")
print(f"Assistant tokens: {assistant_tokens}")
print(f"Other tokens: {total_tokens - assistant_tokens}")

╭───────────────────────────────────────────────── Decoded Text ──────────────────────────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ # Tools / Ferramentas                                                                                           │
│                                                                                                                 │
│ Você pode chamar uma ou mais funções para auxiliar na consulta do usuário.                                      │
│                                                                                                                 │
│ Você recebe assinaturas de funções dentro de tags XML <tools></tools>:                                          │
│ <tools>                                                                                                         │
│ {"type": "function", "function": {"name": "get_weather", "description": "Obter o clima atual", "parameters":    │
│ {"type": "object", "properties": {"location": {"type": "string", "description": "A cidade e o estado"}},        │
│ "required": ["location"]}}}                                                                                     │
│ </tools>                                                                                                        │
│                                                                                                                 │
│ Para cada chamada de função, retorne um objeto json com o nome da função e os argumentos dentro das tags XML    │
│ <tool_call></tool_call>:                                                                                        │
│ <tool_call>                                                                                                     │
│ {"name": <function-name>, "arguments": <args-json-object>}                                                      │
│ </tool_call><|im_end|>                                                                                          │
│ <|im_start|>user                                                                                                │
│ Olá! Como você está?<|im_end|>                                                                                  │
│ <|im_start|>assistant                                                                                           │
│ Estou bem, obrigado! Como posso ajudá-lo hoje?<|im_end|>                                                        │
│ <|im_start|>user                                                                                                │
│ Como está o tempo hoje em Porto Alegre, RS?<|im_end|>                                                           │
│ <|im_start|>assistant                                                                                           │
│ <tool_call>                                                                                                     │
│ {"name": "get_weather", "arguments": {"location": "Porto Alegre, RS"}}                                          │
│ </tool_call><|im_end|>                                                                                          │
│ <|im_start|>user                                                                                                │
│ <tool_response>                                                                                                 │
│ {"temperature": "22°C", "condition": "Ensolarado"}                                                              │
│ </tool_response><|im_end|>                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <think>A chamada para a função get_weather retornou os seguintes dados: temperatura de 22°C e condição          │
│ ensolarada.</think> O tempo em Porto Alegre, RS hoje e

╭─────────────────────────────── Token Visualization (Green=Assistant, Gray=Other) ───────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ # Tools / Ferramentas                                                                                           │
│                                                                                                                 │
│ Você pode chamar uma ou mais funções para auxiliar na consulta do usuário.                                      │
│                                                                                                                 │
│ Você recebe assinaturas de funções dentro de tags XML <tools></tools>:                                          │
│ <tools>                                                                                                         │
│ {"type": "function", "function": {"name": "get_weather", "description": "Obter o clima atual", "parameters":    │
│ {"type": "object", "properties": {"location": {"type": "string", "description": "A cidade e o estado"}},        │
│ "required": ["location"]}}}                                                                                     │
│ </tools>                                                                                                        │
│                                                                                                                 │
│ Para cada chamada de função, retorne um objeto json com o nome da função e os argumentos dentro das tags XML    │
│ <tool_call></tool_call>:                                                                                        │
│ <tool_call>                                                                                                     │
│ {"name": <function-name>, "arguments": <args-json-object>}                                                      │
│ </tool_call><|im_end|>                                                                                          │
│ <|im_start|>user                                                                                                │
│ Olá! Como você está?<|im_end|>                                                                                  │
│ <|im_start|>assistant                                                                                           │
│ Estou bem, obrigado! Como posso ajudá-lo hoje?<|im_end|>                                                        │
│ <|im_start|>user                                                                                                │
│ Como está o tempo hoje em Porto Alegre, RS?<|im_end|>                                                           │
│ <|im_start|>assistant                                                                                           │
│ <tool_call>                                                                                                     │
│ {"name": "get_weather", "arguments": {"location": "Porto Alegre, RS"}}                                          │
│ </tool_call><|im_end|>                                                                                          │
│ <|im_start|>user                                                                                                │
│ <tool_response>                                                                                                 │
│ {"temperature": "22°C", "condition": "Ensolarado"}                                                              │
│ </tool_response><|im_end|>                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <think>A chamada para a função get_weather retornou os seguintes dados: temperatura de 22°C e condição          │
│ ensolarada.</think> O tempo em Porto Alegre, RS hoje e


Total tokens: 321
Assistant tokens: 90
Other tokens: 231
